# Этап 1: DDIM baseline на CIFAR-10

Готовая DDPM, фиксированные шумы, FID, NFE и время. Обучение модели не выполняется. Для Colab требуется среда с GPU; для Kaggle — GPU и доступ к интернету. Код и отчёт: https://github.com/wVks/cv_diff_model.


In [ ]:
import os
import subprocess
from pathlib import Path

project = Path("/content/cv_diff_model") if Path("/content").exists() else Path("/kaggle/working/cv_diff_model") if Path("/kaggle/working").exists() else Path.cwd() / "cv_diff_model"
if not (project / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/wVks/cv_diff_model.git", str(project)], check=True)
os.chdir(project)
print(project)


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt"], check=True)


После установки библиотек может потребоваться перезапуск ядра ноутбука, если torch или numpy уже импортировались. Результаты записываются в отдельный каталог `artifacts/gpu_baseline`, чтобы не смешивать их с локальным прогоном.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Требуется GPU runtime"
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


In [ ]:
subprocess.run([sys.executable, "-u", "-m", "stage1", "benchmark", "--device", "cuda", "--output", "artifacts/gpu_baseline"], check=True)


In [ ]:
subprocess.run([sys.executable, "-u", "-m", "stage1", "run", "--device", "cuda", "--output", "artifacts/gpu_baseline"], check=True)


In [ ]:
import json
from IPython.display import display, Image
results = json.loads(Path("artifacts/gpu_baseline/results.json").read_text())
print(json.dumps(results, ensure_ascii=False, indent=2))
display(Image(filename="artifacts/gpu_baseline/generated_grid.png"))


PDF пересобирается по фактическим результатам. Визуальная заметка в `report/visual_analysis.txt` должна соответствовать показанной сетке; при изменении сэмплера или настроек она проверяется заново. Исходные метрики MPS остаются в Git отдельным прогоном.


In [ ]:
subprocess.run([sys.executable, "scripts/verify_fid.py", "--output", "artifacts/gpu_baseline"], check=True)
subprocess.run([sys.executable, "scripts/build_report.py", "--results", "artifacts/gpu_baseline/results.json"], check=True)


In [ ]:
import shutil
archive = shutil.make_archive("stage1_gpu_results", "zip", root_dir=".", base_dir="artifacts/gpu_baseline")
print(archive)
try:
    from google.colab import files
    files.download(archive)
    files.download("report/stage1_report.pdf")
except ImportError:
    print("Архив и PDF доступны в файловой панели платформы.")
